THIS IS AN OLD MODEL USED TO SET UP THE MODEL ARCHETECTURE 

In [ ]:
import sys
import os

import sys
import os

# import system libs
import os
import time
import shutil
import pathlib
import itertools
import random
import warnings

# import data handling tools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.applications import Xception
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras import regularizers

import sys
import os





# set styles and filter warnings
sns.set_style('darkgrid')
warnings.filterwarnings("ignore")


# Base path relative to notebook location
base_dir = os.path.abspath(os.path.join("..", "archive"))

# Add the parent directory of `pipeline_helpers` to the Python path
sys.path.append(os.path.abspath(".."))
from pipeline_helpers.data_preprocessing import prepare_generators


train_gen, val_gen, test_gen, class_dict = prepare_generators(base_dir)


def build_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape, pooling='max')
    base_model.trainable = False

    model = Sequential([
        base_model,
        Flatten(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer=Adamax(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', Precision(), Recall()])
    return model

# Add the root directory to sys.path
sys.path.append(os.path.abspath('..'))

print("\n Training model on train/validation split")



# Build the model
model = build_model()

# Train the model
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)


from pipeline_helpers.evaluation import (
    save_model_architecture,
    save_model_weights,
    plot_training_metrics,
    evaluate_and_save_results,
    save_confusion_matrix,
    predict_and_save_plot
)


output_dir = '../outputs/models/light_model_evaluation'
os.makedirs(output_dir, exist_ok=True)
# Save model architecture diagram
save_model_architecture(model, output_dir, 'architecture.png')

# Save model weights
save_model_weights(model, output_dir, 'light_model')

# Plot training metrics
plot_training_metrics(history, os.path.join(output_dir, 'training_metrics.png'))

# Evaluate on all three sets
evaluate_and_save_results(model, train_gen, val_gen, test_gen, output_dir)

# Confusion matrix (for test set)
class_labels = list(train_gen.class_indices.keys())  # Get class names
save_confusion_matrix(model, test_gen, class_labels, os.path.join(output_dir, 'confusion_matrix.png'))
